# Order schedule — merge pipeline (notebook)

Reads the datastore through **Forge Anvil's DataWorkbench** (Azure, `customer_name="tbretail"`) — no Spark session or `customer-pipeline-tbretail` checkout needed. Run this notebook under `local_forge_venv`.

**Two ways to get `jira_downloads/cleaned/{KEY}_cleaned.csv` + `pipeline_meta.json`:**

1. **Terminal:** `python scripts/jira_input.py <KEY>` (Jira REST or `--excel-path`), then run the cells below.
2. **Manual (online / no Excel):** Set `MANUAL_SUBMISSION_CSV` in the next cell to a CSV with columns  
   `order_group_description`, `destination_code`, `order_schedule_date`, `order_frequency`, `order_review_calendar`.  
   Optionally set `MANUAL_ISSUE_SUMMARY` / `MANUAL_ISSUE_DESCRIPTION` for fiscal inference.

**Then run the last cell** — it delegates to `order_sch_main.py` (same logic as `python scripts/run_pipeline.py` after `jira_input`).

**Never write to the datastore** (read-only prod + fiscal calendar).


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# EDIT THESE
# ---------------------------------------------------------------------------
JIRA_ISSUE_KEY = "TBRCS-659"

# If you already ran jira_input.py, leave this as None and the notebook uses the cleaned CSV on disk.
# Otherwise set to a CSV path to load, clean, and write {KEY}_cleaned.csv + pipeline_meta.json here.
MANUAL_SUBMISSION_CSV: Path | None = None  # e.g. Path.home() / "draft_order_schedule.csv"

# Used only when MANUAL_SUBMISSION_CSV is set (paste from Jira for fiscal detection)
MANUAL_ISSUE_SUMMARY = ""
MANUAL_ISSUE_DESCRIPTION = ""

# None = infer from text + data; True/False = force fiscal flag in pipeline_meta
FISCAL_OVERRIDE: bool | None = None

# If the notebook cannot find the toolkit, set explicitly (clone root containing `scripts/`)
TOOLKIT_ROOT_OVERRIDE: Path | None = None
# ---------------------------------------------------------------------------

def _resolve_toolkit_root() -> Path:
    if TOOLKIT_ROOT_OVERRIDE is not None:
        return TOOLKIT_ROOT_OVERRIDE.expanduser().resolve()
    here = Path.cwd().resolve()
    if (here / "scripts" / "toolkit_env.py").is_file():
        return here
    if (here / "toolkit_env.py").is_file():
        return here.parent
    raise FileNotFoundError(
        "Could not find toolkit root (directory containing scripts/toolkit_env.py). "
        "Open the notebook from the order-schedule-toolkit repo or set TOOLKIT_ROOT_OVERRIDE."
    )

TOOLKIT_ROOT = _resolve_toolkit_root()
SCRIPTS = TOOLKIT_ROOT / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from toolkit_env import bootstrap_toolkit_env

bootstrap_toolkit_env()

JIRA_ISSUE_KEY = str(JIRA_ISSUE_KEY).strip().upper()

import pandas as pd

if MANUAL_SUBMISSION_CSV is not None:
    import jira_input as ji

    mc = Path(MANUAL_SUBMISSION_CSV).expanduser().resolve()
    if not mc.is_file():
        raise FileNotFoundError(mc)
    df0 = pd.read_csv(mc)
    cleaned, _logs = ji.clean_submission_dataframe(df0)
    cleaned_dir = TOOLKIT_ROOT / "jira_downloads" / "cleaned"
    cleaned_dir.mkdir(parents=True, exist_ok=True)
    cleaned_path = cleaned_dir / f"{JIRA_ISSUE_KEY}_cleaned.csv"
    cleaned.to_csv(cleaned_path, index=False)

    if FISCAL_OVERRIDE is True:
        pipeline_flags = {
            "is_fiscal": True,
            "fiscal_anchor_weekday": 0,
            "rationale": "notebook FISCAL_OVERRIDE=True",
            "explicit_non_fiscal": False,
        }
    elif FISCAL_OVERRIDE is False:
        pipeline_flags = {
            "is_fiscal": False,
            "fiscal_anchor_weekday": 0,
            "rationale": "notebook FISCAL_OVERRIDE=False",
            "explicit_non_fiscal": False,
        }
    else:
        text_flags = ji.infer_fiscal_from_jira_text(MANUAL_ISSUE_SUMMARY, MANUAL_ISSUE_DESCRIPTION)
        if text_flags.get("explicit_non_fiscal"):
            pipeline_flags = text_flags
        else:
            from_freq = ji.infer_fiscal_from_order_frequency_column(
                cleaned,
                summary=MANUAL_ISSUE_SUMMARY,
                description_plain=MANUAL_ISSUE_DESCRIPTION,
            )
            pipeline_flags = from_freq if from_freq is not None else text_flags

    pf_meta = {k: v for k, v in pipeline_flags.items() if k in ("is_fiscal", "fiscal_anchor_weekday", "rationale")}
    pipeline_meta = {
        "issue_key": JIRA_ISSUE_KEY,
        "is_fiscal": bool(pf_meta["is_fiscal"]),
        "fiscal_anchor_weekday": int(pf_meta["fiscal_anchor_weekday"]),
        "rationale": pf_meta["rationale"],
        "cleaned_csv": str(cleaned_path.relative_to(TOOLKIT_ROOT)),
        "source_attachments": [mc.name],
    }
    meta_path = cleaned_dir / f"{JIRA_ISSUE_KEY}_pipeline_meta.json"
    meta_path.write_text(json.dumps(pipeline_meta, indent=2), encoding="utf-8")
    print(f"Wrote manual submission -> {cleaned_path} ({len(cleaned)} rows)")
    print(f"Wrote pipeline meta -> {meta_path}")
else:
    print("MANUAL_SUBMISSION_CSV is None — using existing files from jira_input.py if present.")
    cp = TOOLKIT_ROOT / "jira_downloads" / "cleaned" / f"{JIRA_ISSUE_KEY}_cleaned.csv"
    print(f"Expected cleaned CSV: {cp} (exists={cp.is_file()})")

print("TOOLKIT_ROOT:", TOOLKIT_ROOT)
print("JIRA_ISSUE_KEY:", JIRA_ISSUE_KEY)
print("Ready: run the next cell to execute the merge (Forge Anvil DataWorkbench + order_sch_main.main).")


## Run merge (Forge Anvil DataWorkbench)

Requires datastore paths used by `order_sch_main.py`. Generates local `output/`, merge reports, and updates `pipeline_meta` (same as CLI).

**Note:** HTML merge report generation fetches Tabulator over HTTPS once.


In [ ]:
import importlib.util
import sys

main_py = TOOLKIT_ROOT / "scripts" / "order_sch_main.py"
spec = importlib.util.spec_from_file_location("order_sch_main", main_py)
osm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(osm)

_argv = sys.argv
sys.argv = [
    "order_sch_main.py",
    "--issue-key",
    JIRA_ISSUE_KEY,
]
try:
    osm.main()
finally:
    sys.argv = _argv
